# RF / HD / EBC: session comparisons

260827; probes are selected below. Native sorting and paired peak alignment are separate cells. Pairwise overlap keeps probe identity. RF results are loaded from `locate_rf`; HD class lists, comparison rows, and statistics are saved; figure export is optional.


In [ ]:
%matplotlib inline
from functools import partial
from pathlib import Path
import json
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display
from Utils.direction_comparison import (
    compare_both_orders, load_rf_profiles, load_hd_profiles, load_ebc_profiles,
    peak_direction_sums, plot_peak_direction_sums,
)
from Utils.ebc_analysis import (
    BANDS12, load_basler_session, rebuild_basler_hd, load_motive_session, compute_circle_ebc,
    plot_circle_coverage, ebc_distance_peaks, distance_correlations, plot_distance_correlations,
)
from Utils.plotting import apply_light_plot_style
apply_light_plot_style()


In [ ]:
root = Path("/mnt/senzailab/Kai/#Recording/m20/260918")
probes = ("A",)
session2, session9, session12 = [root / f"260918_{n}" for n in (2, 9, 11)]
output_dir = Path("output/hd_rf_ebc_comparison")
output_dir.mkdir(parents=True, exist_ok=True)
save_figures = False
rf_response_window_s = (0.0, 0.2)
rf_smoothing_bins = 1.5
hd_shuffles = 1000
statistics_permutations = 10_000
bootstraps = 2000
seed = 0

def show(figure, name):
    if save_figures:
        figure.savefig(output_dir / f"{name}.png", dpi=180, facecolor="white", transparent=False)
    display(figure)
    plt.close(figure)


## Shared inputs and HD12 rebuild

Run `locate_rf` first for each regular RF map. RF2 reads the saved same-name `.npz`; use the same response window here.

HD9 uses its saved 180-bin counts and occupancy. HD12 is recalculated from current Basler CSV frame IDs, active-low exposure midpoints, and the baseline epoch. Classes are assigned independently within each session.

Comparison coordinates use `(-HD + 180) % 360 - 180`. EBC comparison curves use the same conversion, while RF retains its signed x positions. Curve values stay attached to their source bins; sorting, plotting, offsets, and direction statistics use these converted coordinates directly. Raw headings used for boundary geometry retain their original convention. HD curve smoothing (3°) and aggregation (30 bins) are unchanged.


In [ ]:
profile_tables = {"RF2": [], "HD9": [], "HD12": []}
profile_tables.update({f"EBC12_{band}": [] for band, _, _ in BANDS12})
rf_center_tables = []
for probe in probes:
    rfmap_file = (
        session2 / "data/rfmapping/good/-100_400_1ms" / f"Probe{probe}"
        / f"regular_unitsSpikeCounts_{session2.name}.rfmap"
    )
    hd9_file = session9 / "data/tuning_curves" / f"Probe{probe}" / "tuning_curves.tc"
    ebc12_dir = session12 / "data/spatial_cells" / f"Probe{probe}" / "baseline"

    rf_profiles, centers = load_rf_profiles(
        rfmap_file, probe=probe, window=rf_response_window_s,
        smoothing_bins=rf_smoothing_bins,
    )
    profile_tables["RF2"].append(rf_profiles)
    rf_center_tables.append(centers)
    profile_tables["HD9"].append(load_hd_profiles(hd9_file, probe=probe))
    basler = load_basler_session(session12, probe=probe)
    hd12_path = rebuild_basler_hd(basler, shuffles=hd_shuffles, seed=seed)
    profile_tables["HD12"].append(load_hd_profiles(hd12_path, probe=probe))
    for band, _, _ in BANDS12:
        ebc_file = ebc12_dir / f"egocentric_rate_map_{band}.rfmap"
        profile_tables[f"EBC12_{band}"].append(load_ebc_profiles(ebc_file, probe=probe))
    print(f"Probe{probe} HD12 timing:", basler["source"]["camera_timing"])

profiles = {name: pd.concat(tables) for name, tables in profile_tables.items()}
rf_centers = pd.concat(rf_center_tables)
rf_centers.to_csv(output_dir / "rf_centers.csv")
display(pd.Series({name: len(table) for name, table in profiles.items()}, name="selected_units"))


In [ ]:
compare = partial(compare_both_orders, profiles,
                  save_dir=output_dir / "figures" if save_figures else None,
                  order_dir=output_dir / "orders")


## 2.1 RF2 ↔ HD9: native sorting

In [ ]:
native_21 = compare("HD9", "RF2")

### 2.1 Peak alignment

For each unit, `offset_deg = -reference_peak_deg` in the signed RF coordinates above. Add that same offset to both curves; retain the unaligned row order. All aligned HD, RF, and EBC curves use 30 bins (12° per bin), including a bin centered at 0°. Periodic linear interpolation preserves missing intervals. The −180° bin is displayed as two halves at the plot edges, so the data contain 30 bins. Saved order CSVs use these same peak and offset coordinates.

For example, HD 246° becomes +114°. Its offset is −114°; an RF peak at +66° then appears at −48°.


In [ ]:
aligned_21 = compare("HD9", "RF2", align=True, previous=native_21)

### 2.1 Sum heatmaps

Keep the reference peak at 0°. Shift the paired curve by **+reference_peak**, so its peak is at `wrap(matched_peak + reference_peak)`, exactly the quantity in the peak-sum histogram below. Offset mode instead shifts the paired curve by **−reference_peak**. Both modes retain the native row order and use 30 bins (12°).

For unit 267, HD9 +114° and RF2 +66° give −48° in the offset RF heatmap and −180° in the sum RF heatmap. The reference HD heatmap remains centered at 0° in both modes.


In [ ]:
summed_21 = compare("HD9", "RF2", align="sum", previous=native_21)

### 2.1 Sum of RF and HD peak directions

Take the peak from each unaligned curve in the signed RF coordinates, then calculate `(HD_peak + RF_peak + 180) % 360 - 180` for each shared unit. +180° and −180° denote the same direction. Each point is one unit; rows retain the HD peak order. The histogram has 30 bins (12° each), and its height is the number of units. The table records both peaks and their sum.


In [ ]:
peak_sums_21 = peak_direction_sums(profiles["HD9"], profiles["RF2"], order=native_21["HD9"]["order"])
peak_sums_21.to_csv(output_dir / "HD9_RF2_peak_sums.csv")
display(peak_sums_21)
show(plot_peak_direction_sums(peak_sums_21, hd_label="HD9", rf_label="RF2"), "hd9_rf2_peak_sums")


## 2.2 RF2 ↔ HD12: native sorting

In [ ]:
native_22 = compare("HD12", "RF2")

### 2.2 Peak alignment

In [ ]:
aligned_22 = compare("HD12", "RF2", align=True, previous=native_22)

### 2.2 Sum heatmaps

Use the same pairs and native row orders. Center the reference peak at 0° and add that reference peak angle to the paired curve, using the shared 30-bin grid.


In [ ]:
summed_22 = compare("HD12", "RF2", align="sum", previous=native_22)

### 2.2 Sum of RF and HD peak directions

Take the peak from each unaligned curve in the signed RF coordinates, then calculate `(HD_peak + RF_peak + 180) % 360 - 180` for each shared unit. +180° and −180° denote the same direction. Each point is one unit; rows retain the HD peak order. The histogram has 30 bins (12° each), and its height is the number of units. The table records both peaks and their sum.


In [ ]:
peak_sums_22 = peak_direction_sums(profiles["HD12"], profiles["RF2"], order=native_22["HD12"]["order"])
peak_sums_22.to_csv(output_dir / "HD12_RF2_peak_sums.csv")
display(peak_sums_22)
show(plot_peak_direction_sums(peak_sums_22, hd_label="HD12", rf_label="RF2"), "hd12_rf2_peak_sums")


## 2.3 HD12 ↔ EBC12 distance bands

Each pair uses the HD12/EBC overlap selected by the current profile tables.


In [ ]:
native_23 = {band: compare("HD12", f"EBC12_{band}") for band, _, _ in BANDS12}

### 2.3 Peak alignment

In [ ]:
aligned_23 = {band: compare("HD12", f"EBC12_{band}", align=True, previous=native_23[band]) for band, _, _ in BANDS12}

### 2.3 Sum heatmaps

Use the same pairs and native row orders. Center the reference peak at 0° and add that reference peak angle to the paired curve, using the shared 30-bin grid.


In [ ]:
summed_23 = {band: compare("HD12", f"EBC12_{band}", align="sum", previous=native_23[band]) for band, _, _ in BANDS12}

## 2.4 Session 9: concentric inner / outer boundaries

The XY extrema define the center. The farthest valid trajectory point defines a 15 cm inner radius; the outer radius is 30 cm using the same scale. Inner uses all distances. Outer is split at 30 cm. Distances beyond the observed support stay missing. Motive HD is saved Z yaw (CCW from world +X), so position rays use that same reference frame.

In [ ]:
circle_profile_tables = {"EBC9_inner": [], "EBC9_outer_0-30": [], "EBC9_outer_30-60": []}
for probe in probes:
    motive = load_motive_session(session9, probe=probe)
    circles = compute_circle_ebc(motive)
    circle_paths = {"EBC9_inner": circles["inner"]["paths"]["full"],
                    "EBC9_outer_0-30": circles["outer"]["paths"]["0-30"],
                    "EBC9_outer_30-60": circles["outer"]["paths"]["30-60"]}
    for name, path in circle_paths.items():
        circle_profile_tables[name].append(load_ebc_profiles(path, probe=probe))
profiles.update({name: pd.concat(tables) for name, tables in circle_profile_tables.items()})
# Session 9 boundary geometry and occupancy are shared by all probes.
show(plot_circle_coverage(motive, circles), "session9_circular_coverage")
pd.DataFrame({"profile": list(profiles), "units": [len(p) for p in profiles.values()]}).to_csv(output_dir / "profile_counts.csv", index=False)

coverage = pd.DataFrame([
    {"boundary": name, "actual_min_cm": item["actual_distance_range"][0],
     "actual_max_cm": item["actual_distance_range"][1],
     "visited_bins": int((item["occupancy"] > 0).sum()), "total_bins": item["occupancy"].size}
    for name, item in circles.items()
])
coverage.to_csv(output_dir / "circle_distance_coverage.csv", index=False)
display(coverage)
np.savez_compressed(output_dir / "circle_occupancy.npz",
    inner=circles["inner"]["occupancy"], outer=circles["outer"]["occupancy"],
    inner_distance_edges=circles["inner"]["distance_edges"],
    outer_distance_edges=circles["outer"]["distance_edges"])


### 2.4 Native sorting

The RF2/HD9 pair is shown in 2.1. Both RF2 and HD9 are compared with each circular-boundary profile.

In [ ]:
native_24 = {(reference, name): compare(reference, name) for reference in ("RF2", "HD9") for name in circle_paths}

### 2.4 Peak alignment

In [ ]:
aligned_24 = {(reference, name): compare(reference, name, align=True, previous=native_24[reference, name]) for reference in ("RF2", "HD9") for name in circle_paths}

### 2.4 Sum heatmaps

Use the same pairs and native row orders. Center the reference peak at 0° and add that reference peak angle to the paired curve, using the shared 30-bin grid.


In [ ]:
summed_24 = {(reference, name): compare(reference, name, align="sum", previous=native_24[reference, name]) for reference in ("RF2", "HD9") for name in circle_paths}

## 2.5 RF center y versus EBC preferred distance

For each unit and band, use the distance coordinate of the largest bin in the full angle×distance map. All three tests use identical RF-significant units. Spearman correlations test monotonic association; lines show medians, not fitted models. The three band p values are FDR-adjusted. Paired bootstrap intervals for correlation differences are exploratory 95% intervals. A numerically largest coefficient alone does not establish a difference between bands.

Repeated distances can make bootstrap correlations undefined. The table reports the defined fraction. For these resamples, interval endpoints conservatively allow the full correlation range (−1 to 1; −2 to 2 for differences), instead of silently dropping them. Constant original distances remain undefined. Scatter marker area and numeric labels show coincident units. The saved session-12 map ends at 28.99 cm; its >16 cm tier retains that original coverage.


In [ ]:
peaks = pd.concat([
    ebc_distance_peaks(
        session12 / "data/spatial_cells" / f"Probe{probe}" / "baseline" / "egocentric_rate_map.rfmap",
        probe=probe,
    )
    for probe in probes
], ignore_index=True)
distance_data, distance_stats, rho_differences = distance_correlations(
    rf_centers, peaks, permutations=statistics_permutations, bootstraps=bootstraps, seed=seed,
)
peaks.to_csv(output_dir / "ebc12_distance_peaks.csv", index=False)
distance_data.to_csv(output_dir / "rf_y_ebc_distance_units.csv")
distance_stats.to_csv(output_dir / "rf_y_ebc_distance_correlations.csv")
rho_differences.to_csv(output_dir / "distance_band_correlation_differences.csv", index=False)
display(distance_stats)
display(rho_differences)
best_band = distance_stats.loc[[b for b, _, _ in BANDS12], "rho"].idxmax()
print(f"Numerically largest rho: {best_band} cm; FDR q={distance_stats.loc[best_band, 'q']:.3g}")
print("Band differences with 95% interval excluding zero:")
display(rho_differences.loc[(rho_differences.ci_low > 0) | (rho_differences.ci_high < 0)])
show(plot_distance_correlations(distance_data, distance_stats), "rf_y_ebc_distance")


In [ ]:
settings = {"date": 260827, "probes": probes, "phase": "baseline", "rf_window_s": rf_response_window_s,
            "hd_shuffles": hd_shuffles,
            "statistics_permutations": statistics_permutations, "bootstraps": bootstraps, "seed": seed,
            "comparison_angle_range_deg": [-180, 180], "hd_ebc_to_rf": "(-angle + 180) % 360 - 180",
            "alignment_offset": "-reference_peak_deg", "sum_matched_offset": "+reference_peak_deg", "alignment_bins": 30, "alignment_grid_step_deg": 12,
            "inner_diameter_cm": 30, "outer_diameter_cm": 60, "outer_bands_cm": [[0,30],[30,60]],
            "circle_center_raw": motive["center_raw"].tolist(), "circle_cm_per_unit": motive["cm_per_unit"]}
(output_dir / "settings.json").write_text(json.dumps(settings, indent=2) + "\n")
print("Results:", output_dir.resolve())
